# Práctica 3 - Inteligencia Artificial

## Tema 1: Metaheurísticas para optimización

### Algoritmos genéticos
### Problema de la mochila

José Luis Ruiz Reina

En este ejercicio veremos la implementación en Python de un algoritmo genético y su aplicación para resolver instancias concretas del problema de la mochila.


In [1]:
import random

Lo que sigue es la definición genérica de un problema de optimización para ser resuelto mediante un algoritmo genético 

In [2]:
class Problema_Genetico(object):
    """ Clase para representar un problema para que sea abordado mediante un
    algoritmo genético general. Consta de los siguientes atributos:
    - genes: lista de posibles genes en un cromosoma
    - longitud_cromosoma: la longitud de los cromosomas (todos son del mismo tamaño)
    - decodifica: método que recibe el genotipo (cromosoma) y devuelve el
      fenotipo (elemento del problema original que el cromosoma representa) 
    - fitness: método de valoración de los cromosomas (actúa sobre el
      genotipo)"""

    def __init__(self, genes, longitud_cromosoma, decodifica, fitness):
        self.genes = genes
        self.longitud_cromosoma = longitud_cromosoma
        self.decodifica = decodifica
        self.fitness = fitness

#### Ejercicio 1

Añadir a la clase anterior, completando las partes que faltan en las siguientes funciones, los principales métodos auxiliares para definir un algoritmo genético:

In [3]:
def poblacion_inicial(self, N):                   # (MEMORIZAR)
    """Dado un número natural N, construye aleatoriamente una población inicial con N cromosomas"""

    return [[random.choice(self.genes) for _ in range(self.longitud_cromosoma)] for _ in range(N)]

def cruza_padres(self, c1, c2):
   """ Operador de cruce en un punto de dos cromosomas c1 y c2"""

   pos = random.randrange(1, len(c1))   # ESTO GENERA UN NUMERO ALEATORIO (entre los valores indicados (del 1 a tamaño c1-1), para que no sean bordes, 0 ni el fin) QUE LO VAMOS A USAR COMO PUNTO DE CORTE
   hijo1 = c1[:pos] + c2[pos:]  # Mezclamos a los hijos por los puntos de corte
   hijo2 = c2[:pos] + c1[pos:]  # Mezclamos a los hijos por los puntos de corte
   return [hijo1, hijo2]

def cruza(self, padres):
   """Dado una población de padres obtiene el resultado de cruzar de dos en dos, en el orden en el que aparecen"""

   hijos = []
   for j in range(0, len(padres), 2):
       hijos.extend(self.cruza_padres(padres[j], padres[j+1])) 
   return hijos

def muta_cromosoma(self, cr, prob):
   """A partir de un cromosoma cr, se obtiene un nuevo cromosoma resultado de
      mutar (con una probabilidad prob), cada gen de cr. Nótese que si prob es bajo, entonces
      es probable que el cromosoma devuelto sea igual a cr"""
   
   cm = cr[:] # una copia
   for i in range(len(cr)):         # Recorremos cada gen del cromosoma
       if random.random() < prob:   # LINEA DE CODIGO DE MEMORIA PARA PODER PONER LA PROBABILIDAD DE UN PORCENTAJE
           cm[i] = random.choice(self.genes)        # Recorre todos los genes del cromosoma y si se cumple prob, pues lo cambia por otro gen raro para poder mutar el cromosoma
   return cm

def muta(self, generacion, prob):
   """Obtiene el resultado de mutar cada cromosoma de población, con probabilidad prob"""
   
   return [self.muta_cromosoma(cr, prob) for cr in generacion]

#### Ejercicio 2

Añadir, igualmente, los siguientes métodos, que definen la selección basada en torneo:

In [4]:
def torneo(self, generacion, k, mejor):
   """ Recibe una población, un número natural k y
       y la función mejor (que puede ser max o min, dependiendo si es un problema de 
       maximización o minimización); y selecciona un cromosoma de la población, 
       mediante un torneo de tamaño k"""
   
   participantes = random.sample(generacion, k)     # Saca k individuos al azar de generacion

   return mejor(participantes, key = self.fitness)  # Nos quedamos con el mejor (max o min) segun el fitness

def selecciona_torneo(self, generacion, k, mejor, n):
   """ Recibe una población, un número natural k, 
       la función mejor (que puede ser max o min, dependiendo si es un problema de 
       maximización o minimización), y un número n; y selecciona n cromosomas de la población, 
       mediante n torneos de tamaño k"""    
   
   return [self.torneo(generacion, k, mejor) for _ in range(n)]

#### Ejercicio 3

Usando la función `binario_a_decimal` que se incluye a continuación, definir un objeto `cuad_gen` que define la representación genética usada en el problema de encontrar el número entero entre 0 y $2^{10}$ que tiene un menor cuadrado (usando lo descrito en las diapositivas, pág. 30) 

In [5]:
def binario_a_decimal(cr):
    return sum(g*(2**i) for (i,g) in enumerate(cr))

In [6]:
# Solución:

cuad_gen = Problema_Genetico(
    [0, 1],          # Como binario_a_decimal tranaja en binario los unicos posibles valores son 0 y 1
    10,             # Para representar hasta 2^10 en binario, necesitas exactamente 10 bits (10 posiciones en tu cromosoma)
    binario_a_decimal,      # Traduce la lista de 1's y 0's al numero real que representan
    lambda cr: binario_a_decimal(cr)**2     # Buscamos el menor cuadrado
)

#### Ejercicio 4

Consideremos la siguiente versión de algoritmo genético:

* parámetros: `N` (tamaño de cada generación), `prop_padres` (proporción de padres), `prob_mutar` (probabilidad de mutación)
* métodos: 
    * condición de terminación: número de generaciones, `nGen`
    * selección: por torneo de tamaño `k` y cruce en un punto
    * cruce: en un punto

```
generacion <- poblacion_inicial(N)
Repetir nGen veces:
    directos <- seleccionar N * (1 - prop_padres) cromosomas de generacion
    padres <- seleccionar N * prop_padres cromosomas de generacion
    hijos <- cruzar los padres usando cruce en un punto
    hijos <- mutar los genes de hijos con probabilidad `prob_mutar`
    generacion <- hijos y directos
Devolver el fenotipo, y el fitness, del mejor cromosoma de generacion
```

La siguiente función `algoritmo genetico` implementa dicha versión. Sólo queda por implementar la función `nueva_poblacion`, que a partir de una generación obtiene la siguiente. Se pide implementar esa función.

In [7]:
def algoritmo_genetico(problema_genetico, N, prop_padres, prob_mutar, nGen, k, mejor):
    """Algoritmo genético siguiendo el esquema descrito en el enunciado que, 
       además, va mostrando por pantalla estadísticas de las valoraciones de cada generación
       Devuelve el fenotipo del mejor cromosoma de la última generación, junto con su fitness.
       Recibe como entrada los siguientes argumentos:
       - problema_genetico: un objeto de la clase Problema_Genetico, con la representación del
         problema de optimización.
       - N: tamaño de cada generación
       - prop_padres: proporción de una generación que se usa como padres       por ejemplo: 0.8, si N = 100, entonces tenemos 80 padres
       - prob_mutar: probabilidad de mutación de un gen
       - nGen: número de generaciones 
       - k: número de cromosomas que intervienen en cada torneo
       - mejor: min o max, dependiendo si el problema es de minimización o de maximización
    """
    n_padres = round(N * prop_padres)                           # Selecciona el numero de cromosomas que van a ser padres
    n_padres = (n_padres if n_padres%2==0 else n_padres-1)      # Hace que el numero de padres sea par para que la funcion de cruzarlos no de error
    n_directos = N - n_padres            # Individuos de la generación actual que son seleccionados para pasar a la siguiente ronda directamente, sin cruzarse ni mutar

    generacion = problema_genetico.poblacion_inicial(N)
    
    for t in range(nGen):
        generacion = nueva_poblacion(problema_genetico, generacion, n_padres, n_directos, prob_mutar, k, mejor)
        mejor_cr = mejor(generacion, key = problema_genetico.fitness)
        mejor_fen = problema_genetico.decodifica(mejor_cr)
        print("Generacion: {0}. Media: {1}. Fitness mejor:{2}".format(t+1, media(problema_genetico.fitness, generacion, N),
                                                                      problema_genetico.fitness(mejor_cr)))        
    return (mejor_fen, problema_genetico.fitness(mejor_cr)) 

def media(fitness, generacion, N):
    return sum(map(fitness, generacion)) / N

In [8]:
# Solución:

def nueva_poblacion(problema_genetico, generacion, n_padres, n_directos, prob_mutar, k, mejor):     # BASICAMENTE ES USAR LOS METODOS CREADOS ANTERIORMENTE
    directos = problema_genetico.selecciona_torneo(generacion, k, mejor, n_directos)    # Seleccion por torneo
    padres = problema_genetico.selecciona_torneo(generacion, k, mejor, n_padres)        # Seleccion por torneo
    hijos = problema_genetico.cruza(padres)                         # Cruzamos
    hijos_mutados = problema_genetico.muta(hijos, prob_mutar)       # Mutamos

    return directos + hijos_mutados     # Devolvemos todos los individuos de nuestra nueva generacion

#### Ejercicio 5

Experimentar y observar con el algoritmo genético anterior cómo evolucionan las generaciones en el problema del mínimo de la función cuadrado que se ha representado en el Ejercicio 3

In [12]:
# COPDIGO DE GEMINI PARA SOLUCIONAR UN PROBLEMA DE LOS CUADERNOS DE JUPYTER AL EJERCURA LA CELDA DE ABAJO


# 1. Pegamos los métodos del Ejercicio 1 a la clase
Problema_Genetico.poblacion_inicial = poblacion_inicial
Problema_Genetico.cruza_padres = cruza_padres
Problema_Genetico.cruza = cruza
Problema_Genetico.muta_cromosoma = muta_cromosoma
Problema_Genetico.muta = muta

# 2. Pegamos los métodos del Ejercicio 2 a la clase
Problema_Genetico.torneo = torneo
Problema_Genetico.selecciona_torneo = selecciona_torneo

In [14]:
algoritmo_genetico(cuad_gen, 20, 0.75, 0.1, 10, 4, min)

Generacion: 1. Media: 78755.2. Fitness mejor:900
Generacion: 2. Media: 20079.65. Fitness mejor:361
Generacion: 3. Media: 5971.35. Fitness mejor:324
Generacion: 4. Media: 15034.8. Fitness mejor:324
Generacion: 5. Media: 18646.15. Fitness mejor:324
Generacion: 6. Media: 25728.65. Fitness mejor:256
Generacion: 7. Media: 14638.45. Fitness mejor:256
Generacion: 8. Media: 28788.0. Fitness mejor:0
Generacion: 9. Media: 30748.75. Fitness mejor:0
Generacion: 10. Media: 27212.8. Fitness mejor:0


(0, 0)

## Problema de la mochila

#### Ejercicio 6

Se dan a continuación tres problemas de la mochila de distinta complejidad. 

Se pide aplicar el algoritmo genético anterior a cada uno de los problemas, y comprobar los resultados que se obtienen (se incluye la solución óptima de cada uno de ellos, para que sirva de referencia sobre el rendimiento de nuestro algoritmo genético). 

In [16]:
# Problema de la mochila 1:
# 10 objetos, peso máximo 165
pesos1 = [23,31,29,44,53,38,63,85,89,82]
valores1 = [92,57,49,68,60,43,67,84,87,72]

# Solución óptima= [1,1,1,1,0,1,0,0,0,0], con valor 309
# ---------------------------------------------------------




# Problema de la mochila 2:
# 15 objetos, peso máximo 750

pesos2 = [70,73,77,80,82,87,90,94,98,106,110,113,115,118,120]
valores2 = [135,139,149,150,156,163,173,184,192,201,210,214,221,229,240]

# Solución óptima= [1,0,1,0,1,0,1,1,1,0,0,0,0,1,1] con valor 1458
# ------------------------------------------------------------




# Problema de la mochila 3:
# 24 objetos, peso máximo 6404180

pesos3 = [382745,799601,909247,729069,467902, 44328,
       34610,698150,823460,903959,853665,551830,610856,
       670702,488960,951111,323046,446298,931161, 31385,496951,264724,224916,169684]
valores3 = [825594,1677009,1676628,1523970, 943972,  97426,
       69666,1296457,1679693,1902996,
       1844992,1049289,1252836,1319836, 953277,2067538, 675367,
       853655,1826027, 65731, 901489, 577243, 466257, 369261]

# Solución óptima= [1,1,0,1,1,1,0,0,0,1,1,0,1,0,0,1,0,0,0,0,0,1,1,1] con valoración 13549094
# --------------------------------------------------------------------





# SOLUCION DE GEMINI

# 1. Definimos una función que nos cree automáticamente un Problema_Genetico para la mochila
def crea_problema_mochila(pesos, valores, peso_max):
    genes = [0, 1]
    longitud = len(pesos)
    
    # El cromosoma ya es la lista de ceros y unos, lo devolvemos tal cual
    decodifica = lambda cr: cr
    
    def fitness(cr):
        # Calculamos el peso y el valor sumando solo los objetos que tienen un 1
        peso_total = sum(cr[i] * pesos[i] for i in range(longitud))
        valor_total = sum(cr[i] * valores[i] for i in range(longitud))
        
        # Penalización letal: si nos pasamos del peso máximo, la solución no vale nada
        if peso_total > peso_max:
            return 0
        return valor_total
        
    return Problema_Genetico(genes, longitud, decodifica, fitness)


# 2. Creamos los 3 problemas con los datos de tu celda
mochila1 = crea_problema_mochila(pesos1, valores1, 165)
mochila2 = crea_problema_mochila(pesos2, valores2, 750)
mochila3 = crea_problema_mochila(pesos3, valores3, 6404180)


# 3. Lanzamos el algoritmo para cada mochila
# Nota: A medida que el problema es más difícil, aumentamos la población (N) 
# y el número de generaciones (nGen) para darle tiempo a encontrar la solución óptima.

print("--- RESULTADOS MOCHILA 1 (10 objetos) ---")
# Usamos 'max' porque queremos MAXIMIZAR el valor
algoritmo_genetico(mochila1, N=50, prop_padres=0.75, prob_mutar=0.1, nGen=50, k=3, mejor=max)

print("\n--- RESULTADOS MOCHILA 2 (15 objetos) ---")
algoritmo_genetico(mochila2, N=100, prop_padres=0.75, prob_mutar=0.1, nGen=100, k=3, mejor=max)

print("\n--- RESULTADOS MOCHILA 3 (24 objetos) ---")
# Aquí le damos una población y tiempo mayores porque el espacio de búsqueda es enorme
algoritmo_genetico(mochila3, N=200, prop_padres=0.8, prob_mutar=0.15, nGen=200, k=5, mejor=max)


--- RESULTADOS MOCHILA 1 (10 objetos) ---
Generacion: 1. Media: 83.72. Fitness mejor:263
Generacion: 2. Media: 114.76. Fitness mejor:277
Generacion: 3. Media: 115.02. Fitness mejor:277
Generacion: 4. Media: 163.9. Fitness mejor:309
Generacion: 5. Media: 201.0. Fitness mejor:309
Generacion: 6. Media: 168.38. Fitness mejor:277
Generacion: 7. Media: 189.14. Fitness mejor:277
Generacion: 8. Media: 196.62. Fitness mejor:277
Generacion: 9. Media: 180.28. Fitness mejor:277
Generacion: 10. Media: 211.7. Fitness mejor:277
Generacion: 11. Media: 205.16. Fitness mejor:277
Generacion: 12. Media: 231.16. Fitness mejor:277
Generacion: 13. Media: 225.88. Fitness mejor:277
Generacion: 14. Media: 211.28. Fitness mejor:277
Generacion: 15. Media: 224.14. Fitness mejor:284
Generacion: 16. Media: 219.94. Fitness mejor:284
Generacion: 17. Media: 214.82. Fitness mejor:284
Generacion: 18. Media: 203.04. Fitness mejor:284
Generacion: 19. Media: 222.68. Fitness mejor:284
Generacion: 20. Media: 230.16. Fitness m

([1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0],
 13467914)